<div style="text-align: justify;">

# Time-Series Simulation and DER Modeling

## Objectives

Chapter 1 solved a network once and read the result back. Real feeders never
sit still: demand and rooftop solar both change every half hour, and a
single snapshot can miss the moment voltage actually goes out of bounds.
This notebook covers the rest of `ark.dss`, the part built for that:

- Turn a real smart-meter reading into an OpenDSS `LoadShape` and drive a
  `Set Mode=daily` simulation with it, not a single solve.
- Model PV properly: an irradiance/temperature-driven `PVSystem`, not a
  flat kVA rating.
- Curtail or reactive-compensate PV output with Volt-Watt and Volt-VAr
  inverter control, and add a dispatchable `Storage` element.
- Run a real PV hosting-capacity study on the same 31-customer AusNet
  feeder Chapter 1 introduced, real smart-meter and PV data, sweeping PV
  penetration until the feeder actually violates its voltage limits.

`Circuit` picked up three new pieces for this chapter: `circuit.pvsystems`
and `circuit.storage_units` (iterators, the same pattern as
`circuit.loads`), and `circuit.solve_daily(steps, stepsize)` (steps through
a time-series solve, one call per half hour instead of one call per
network).

</div>

<div style="text-align: justify;">

## Getting the data

This notebook needs Team-Nando's real AusNet LV network and smart-meter
data, the same 31-customer feeder Chapter 1 introduced. It isn't checked
into the repository (it's someone else's licensed data, vendored locally
instead), so fetch it once:

```bash
uv run python scripts/fetch_part4_ausnet_data.py
```

That downloads `LVcircuit-*.txt` (the network model) and two `.npy` files,
342 real, anonymized customers' active power and normalized PV generation,
both at 30-minute resolution for a full year, into
`resources/cvar_flexibility/data/timeseries-lv/`.

</div>

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from ark.dss.circuit import Circuit

DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")

<div style="text-align: justify;">

## 1. From one reading to a year of them

Start from the same three-house network Chapter 1 built by hand (Figure 1
there): a transformer, one three-phase backbone, three single-phase houses.
Wrapping it in a function here, same as Chapter 1's own exercises did, so
every section below can rebuild a clean copy instead of reusing one
circuit's accumulated state.

</div>

In [ ]:
def build_simple_lv_network() -> Circuit:
    circuit = Circuit()
    circuit.command("Clear")
    circuit.command("Set DefaultBaseFrequency=50")
    circuit.command("New Circuit.Simple_LV_Network")
    circuit.command("Edit vsource.source bus1=sourceBus basekv=11 pu=1.0 phases=3")
    circuit.command(
        "New transformer.LVTR Buses=[sourcebus, A.1.2.3] Conns=[delta wye] "
        "KVs=[11, 0.4] KVAs=[250 250] %Rs=0.00 xhl=2.5 %loadloss=0"
    )
    circuit.command("new linecode.240sq nphases=3 R1=0.127 X1=0.072 R0=0.342 X0=0.089 units=km")
    circuit.command("new linecode.16sq nphases=1 R1=1.15 X1=0.083 R0=1.2 X0=0.083 units=km")
    circuit.command("new line.A_B bus1=A.1.2.3 bus2=B.1.2.3 length=1 phases=3 units=km linecode=240sq")
    circuit.command("new line.B_L1 bus1=B.1 bus2=C.1 length=0.01 phases=1 units=km linecode=16sq")
    circuit.command("new line.B_L2 bus1=B.2 bus2=D.1 length=0.01 phases=1 units=km linecode=16sq")
    circuit.command("new line.B_L3 bus1=B.3 bus2=E.1 length=0.01 phases=1 units=km linecode=16sq")
    circuit.command(
        "new load.Load_1 bus1=C.1 phases=1 kV=(0.4 3 sqrt /) kW=7 pf=0.95 "
        "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=fixed"
    )
    circuit.command(
        "new load.Load_2 bus1=D.1 phases=1 kV=(0.4 3 sqrt /) kW=6 pf=0.95 "
        "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=fixed"
    )
    circuit.command(
        "new load.Load_3 bus1=E.1 phases=1 kV=(0.4 3 sqrt /) kW=8 pf=0.95 "
        "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=fixed"
    )
    circuit.command("set controlmode=static")
    circuit.command("Set VoltageBases=[11 0.4]")
    circuit.command("calcvoltagebases")
    return circuit

<div style="text-align: justify;">

### 1.1 A real smart-meter reading

`Residential load data 30-min resolution.npy` holds one year of real,
anonymized active-power readings for 342 AusNet customers, 48 half-hour
values per day. This is the actual output of a smart meter: a load, at a
bus, over time, exactly the thing a `LoadShape` is built to carry.

</div>

In [ ]:
load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
pv_data = np.load(DATA_DIR / "Residential PV data 30-min resolution.npy")

print(f"load_data: {load_data.shape} (customers, days, half-hours)")
print(f"pv_data:   {pv_data.shape} (days, half-hours), one normalised profile per day")

load_data: (342, 365, 48) (customers, days, half-hours)
pv_data:   (365, 48) (days, half-hours), one normalised profile per day


<div style="text-align: justify;">

### 1.2 Building a LoadShape from it

A meter only ever reports active power (kW); `qmult` (reactive power) isn't
measured, so it's derived from an assumed power factor, the same
`0.95`-`0.99` range Team-Nando's own tutorial uses. `useactual=1` tells
OpenDSS the `pmult`/`qmult` arrays are already in kW/kvar, not multipliers
on the load's own rated `kW`.

`edit load.Load_1 daily=customer0_profile` is the whole bridge: from here
on, every `Set Mode=daily` solve reads `Load_1`'s demand from this shape
instead of its fixed `kW`.

</div>

In [ ]:
rng = np.random.default_rng(42)

customer_idx, day_idx = 0, 45
customer_profile = load_data[customer_idx, day_idx, :]
power_factor = rng.uniform(0.95, 0.99, len(customer_profile))
apparent_profile = customer_profile / power_factor
reactive_profile = np.sqrt(np.maximum(apparent_profile**2 - customer_profile**2, 0))

circuit = build_simple_lv_network()
circuit.command(
    f"New LoadShape.customer0_profile npts=48 minterval=30 useactual=1 "
    f"pmult={customer_profile.tolist()} qmult={reactive_profile.tolist()}"
)
circuit.command("edit load.Load_1 daily=customer0_profile")
print(f"Load_1 now follows customer {customer_idx}'s real reading for day {day_idx}")

Load_1 now follows customer 0's real reading for day 45


<div style="text-align: justify;">

Plotting the shape directly confirms it: a typical residential profile,
low overnight, a small midday bump, a sharp evening peak past 3kW.

</div>

In [ ]:
from lets_plot import LetsPlot, aes, geom_line, ggplot, ggsize, labs

from ark.plot.theme import modern_theme
from ark.plot.tokens import PRIMARY

LetsPlot.setup_html()

half_hours = pd.DataFrame({"half_hour": range(48), "kw": customer_profile})
(
    ggplot(half_hours, aes(x="half_hour", y="kw"))
    + geom_line(color=PRIMARY, size=1)
    + labs(x="Half-hour of day", y="Active power (kW)", title=f"Customer {customer_idx}, day {day_idx}")
    + modern_theme()
    + ggsize(650, 320)
)

<div style="text-align: justify;">

## 2. Solving across time

`circuit.solve_daily(steps, stepsize)` sets `Mode=daily` and steps through
one solve per interval, the LoadShape supplying a different reading at each
step instead of one fixed value. This is exactly what Chapter 1's
`.ark-mistake` callout was setting up: a single snapshot can't show this,
only a real time series can.

</div>

In [ ]:
voltage_over_day = []
for step in circuit.solve_daily(steps=48):
    vmag_pu = circuit.bus_voltages().query("bus == 'c'")["vmag_pu"].iloc[0]
    voltage_over_day.append({"half_hour": step, "vmag_pu": vmag_pu})

voltage_over_day = pd.DataFrame(voltage_over_day)
print(f"Converged: {circuit.converged}")
vmin, vmax = voltage_over_day["vmag_pu"].min(), voltage_over_day["vmag_pu"].max()
print(f"Voltage at house C ranges {vmin:.4f}-{vmax:.4f} pu over the day")

Converged: True
Voltage at house C ranges 0.9768-0.9768 pu over the day


In [ ]:
from lets_plot import ylim

from ark.plot.tokens import INFO

# A fixed axis spanning the statutory band, not an auto-scaled one: the
# real range here is 0.000003 pu, and letting the axis zoom to that would
# draw a dramatic-looking cliff out of what the text above already says
# is noise. This range is what "barely moves" actually looks like next to
# the compliance band that matters.
(
    ggplot(voltage_over_day, aes(x="half_hour", y="vmag_pu"))
    + geom_line(color=INFO, size=1)
    + ylim(0.94, 1.10)
    + labs(x="Half-hour of day", y="Voltage (pu)", title="House C voltage, driven by customer 0's real load")
    + modern_theme()
    + ggsize(650, 320)
)

<div style="text-align: justify;">

Voltage barely moves: 0.976820 to 0.976823 pu, a swing of about
0.000003 pu, nothing a real meter would ever register. That's a real result
too, not a failed demo: Figure 1's backbone (a 250kVA transformer feeding a
short, heavy 240sq three-phase line) is strong relative to one house's
ordinary swing between roughly 0.3kW overnight and 3kW at its evening peak.
A network's strength matters as much as a load's size; the mechanism this
section demonstrates, solving across time instead of once, is real even
when this particular feeder barely shows it. Section 6 puts real demand
from 31 real customers on a thinner, real feeder, and there it does.

</div>

<div style="text-align: justify;">

## 3. Modeling PV properly

A `PVSystem` can run on a flat rating (`kva`/`pmpp`, constant output), the
same way Chapter 1's exercises added PV. A more realistic model drives it
from an irradiance profile and a temperature profile instead, a `LoadShape`
for sunlight and a `Tshape` for panel temperature, both attached via
`daily=`/`Tdaily=`.

</div>

In [ ]:
# Illustrative daily irradiance (fraction of 1kW/m^2) and panel temperature
# (degrees C), not real weather data: real AusNet data has normalised PV
# *generation* profiles (used directly in Section 6's hosting-capacity
# study), not separate irradiance/temperature series.
irradiance = [0, 0, 0, 0, 0, 0, 0.1, 0.2, 0.3, 0.5, 0.8, 0.9, 1.0, 1.0, 0.99, 0.9, 0.7, 0.4, 0.1, 0, 0, 0, 0, 0]
temperature = [15, 15, 15, 15, 15, 15, 16, 18, 20, 23, 26, 29, 31, 32, 31, 28, 24, 20, 17, 16, 15, 15, 15, 15]


def pv_export_by_hour(circuit: Circuit) -> list[float]:
    circuit.command(f"New LoadShape.irradiance_C npts=24 minterval=60 mult={irradiance}")
    circuit.command(f"New Tshape.temp_C npts=24 minterval=60 temp={temperature}")
    circuit.command("edit PVSystem.PV_C daily=irradiance_C Tdaily=temp_C")
    return [-circuit.element_powers("pvsystems")["p_kw"].iloc[0] for _ in circuit.solve_daily(steps=24, stepsize="1h")]


default_cutin = build_simple_lv_network()
default_cutin.command("new PVSystem.PV_C bus1=C.1 phases=1 kva=5 pmpp=5 pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1")
pv_export_default = pv_export_by_hour(default_cutin)
print("PV export by hour, OpenDSS's default %cutin/%cutout:", [round(p, 2) for p in pv_export_default])

PV export by hour, OpenDSS's default %cutin/%cutout: [np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(1.0), np.float64(1.5), np.float64(2.5), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.0), np.float64(4.95), np.float64(4.5), np.float64(3.5), np.float64(2.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0)]


<div style="text-align: justify;">

::: {.ark-mistake}

**A 20% default nearly zeroed out the morning and evening.** OpenDSS's
`PVSystem` defaults `%cutin`/`%cutout` to 20%, the inverter output threshold
below which it won't turn on or stays off. At 10% irradiance (hour 6 above),
a real inverter still produces something, `0.1 x 5kW = 0.5kW`, but a
20%-cutin model reports flat `0.0`, and it's easy to miss unless you check
hour-by-hour: hours 12-16, right in the middle of the day, look identical
either way. Team-Nando's own tutorials always override this to
`%cutin=0.05 %cutout=0.05`; rebuilding the same PV system with that override
recovers the missing hours.

:::

</div>

In [ ]:
fixed_cutin = build_simple_lv_network()
fixed_cutin.command(
    "new PVSystem.PV_C bus1=C.1 phases=1 kva=5 pmpp=5 pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 "
    "%cutin=0.05 %cutout=0.05"
)
pv_export_fixed = pv_export_by_hour(fixed_cutin)
print("PV export by hour, %cutin/%cutout=5%:      ", [round(p, 2) for p in pv_export_fixed])

PV export by hour, %cutin/%cutout=5%:       [np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.5), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.0), np.float64(4.95), np.float64(4.5), np.float64(3.5), np.float64(2.0), np.float64(0.5), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0), np.float64(-0.0)]


In [ ]:
from lets_plot import scale_color_manual

from ark.plot.tokens import WARNING

hours = pd.DataFrame(
    {
        "hour": list(range(24)) * 2,
        "kw": pv_export_default + pv_export_fixed,
        "cutin": ["20% (default)"] * 24 + ["5% (Team-Nando's own setting)"] * 24,
    }
)
(
    ggplot(hours, aes(x="hour", y="kw", color="cutin"))
    + geom_line(size=1)
    + scale_color_manual(values=[WARNING, PRIMARY])
    + labs(x="Hour of day", y="PV export (kW)", title="The default cutin threshold silently drops real output")
    + modern_theme()
    + ggsize(650, 320)
)

<div style="text-align: justify;">

## 4. Mitigation strategies: Volt-Watt and Volt-VAr

A big rooftop system on a weak feeder connection can push local voltage
above the statutory limit long before the transformer itself is stressed.
Volt-Watt curtails PV output as voltage rises past a threshold; Volt-VAr
absorbs reactive power instead, trading real-power export for headroom.
Both are `InvControl` elements driven by an `XYCurve` of voltage/output
setpoints.

The network below is deliberately a stress case, one house with a 10kVA
system on a long, thin service cable, to make the effect visible without
guesswork about whether it would ever actually trigger.

</div>

In [ ]:
def build_pv_stress_network(pv_kva: float = 10.0) -> Circuit:
    circuit = Circuit()
    circuit.command("Clear")
    circuit.command("Set DefaultBaseFrequency=50")
    circuit.command("New Circuit.PV_Stress_Network")
    circuit.command("Edit vsource.source bus1=sourceBus basekv=11 pu=1.0 phases=3")
    circuit.command(
        "New transformer.LVTR Buses=[sourcebus, A.1.2.3] Conns=[delta wye] "
        "KVs=[11, 0.4] KVAs=[250 250] %Rs=0.00 xhl=2.5 %loadloss=0"
    )
    circuit.command("new linecode.16sq nphases=1 R1=1.15 X1=0.083 R0=1.2 X0=0.083 units=km")
    circuit.command("new line.A_C bus1=A.1 bus2=C.1 length=0.5 phases=1 units=km linecode=16sq")
    circuit.command(
        "new load.Load_1 bus1=C.1 phases=1 kV=(0.4 3 sqrt /) kW=1 pf=0.95 "
        "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=fixed"
    )
    circuit.command(
        f"new PVSystem.PV_C bus1=C.1 phases=1 kva={pv_kva} pmpp={pv_kva} pf=1 "
        "kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
    )
    circuit.command("Set VoltageBases=[11 0.4]")
    circuit.command("calcvoltagebases")
    circuit.command("set controlmode=static")
    circuit.command("set mode=snapshot")
    # InvControl needs more than the default 500 control iterations to
    # settle; without this, the solve raises "Max Control Iterations
    # Exceeded" instead of converging, caught by running this cell, not by
    # reading the OpenDSS manual.
    circuit.command("Set maxcontroliter=1000")
    return circuit


_demo = build_pv_stress_network()
_demo.solve()
print(_demo.summary())

{'converged': True, 'n_buses': 3, 'n_lines': 1, 'n_loads': 1, 'n_transformers': 1, 'total_p_loss_kw': 0.7376729775028259, 'total_q_loss_kvar': 0.0739445081686662}


In [ ]:
no_control = build_pv_stress_network()
no_control.solve()
v_no_control = no_control.bus_voltages().query("bus == 'c'")["vmag_pu"].iloc[0]
p_no_control = -no_control.element_powers("pvsystems")["p_kw"].iloc[0]

volt_watt = build_pv_stress_network()
volt_watt.command("New XYCurve.vw_curve npts=4 Xarray=(0.5, 1.05, 1.10, 1.5) Yarray=(1.0, 1.0, 0.0, 0.0)")
volt_watt.command(
    "New InvControl.InvPVCtrl mode=VOLTWATT voltage_curvex_ref=rated voltwatt_curve=vw_curve "
    "DeltaP_factor=0.1 voltagechangetolerance=0.0001 varchangetolerance=0.1 EventLog=no"
)
volt_watt.solve()
v_volt_watt = volt_watt.bus_voltages().query("bus == 'c'")["vmag_pu"].iloc[0]
p_volt_watt = -volt_watt.element_powers("pvsystems")["p_kw"].iloc[0]

print(f"No control:  voltage={v_no_control:.4f} pu, PV export={p_no_control:.2f} kW")
print(f"Volt-Watt:   voltage={v_volt_watt:.4f} pu, PV export={p_volt_watt:.2f} kW")

No control:  voltage=1.0887 pu, PV export=10.00 kW
Volt-Watt:   voltage=1.0637 pu, PV export=7.32 kW


In [ ]:
from lets_plot import geom_bar, scale_fill_manual

from ark.plot.tokens import WARNING

comparison = pd.DataFrame(
    {
        "scenario": ["No control", "Volt-Watt"],
        "vmag_pu": [v_no_control, v_volt_watt],
        "pv_export_kw": [p_no_control, p_volt_watt],
    }
)
(
    ggplot(comparison, aes(x="scenario", y="pv_export_kw", fill="scenario"))
    + geom_bar(stat="identity", width=0.5)
    + scale_fill_manual(values=[PRIMARY, WARNING])
    + labs(x="", y="PV export (kW)", title="Volt-Watt curtails export as voltage rises")
    + modern_theme()
    + ggsize(420, 320)
)

<div style="text-align: justify;">

Curtailing real power is one lever; Volt-VAr pulls a different one,
absorbing reactive power instead of cutting generation, and the inverter's
own `kVA` rating couples the two: more `Q` leaves less headroom for `P`
within the same apparent-power limit.

</div>

In [ ]:
volt_var = build_pv_stress_network()
volt_var.command("New XYCurve.vv_curve npts=4 Xarray=(0.5, 0.95, 1.05, 1.5) Yarray=(1.0, 1.0, -1.0, -1.0)")
volt_var.command("New InvControl.InvPVCtrlVV mode=VOLTVAR voltage_curvex_ref=rated vvc_curve1=vv_curve EventLog=no")
volt_var.solve()

v_volt_var = volt_var.bus_voltages().query("bus == 'c'")["vmag_pu"].iloc[0]
pq_volt_var = volt_var.element_powers("pvsystems")[["p_kw", "q_kvar"]].iloc[0]

print(f"Volt-VAr: voltage={v_volt_var:.4f} pu, PV P={-pq_volt_var['p_kw']:.2f} kW, Q={-pq_volt_var['q_kvar']:.2f} kvar")
print(
    f"P dropped from {p_no_control:.2f} kW to {-pq_volt_var['p_kw']:.2f} kW even though Volt-VAr only "
    "commands Q: sqrt(P^2+Q^2) is capped at the inverter's 10kVA rating, so absorbing reactive power "
    "leaves less headroom for real power within that same limit."
)

Volt-VAr: voltage=1.0662 pu, PV P=8.35 kW, Q=-5.50 kvar
P dropped from 10.00 kW to 8.35 kW even though Volt-VAr only commands Q: sqrt(P^2+Q^2) is capped at the inverter's 10kVA rating, so absorbing reactive power leaves less headroom for real power within that same limit.


<div style="text-align: justify;">

## 5. Storage

A `Storage` element is a dispatchable battery: `state=CHARGING` draws power
from the network, `state=DISCHARGING` supplies it, both bounded by
`kwrated` (power) and `kwhrated`/`kwhstored` (energy). Unlike a `Load` or a
`PVSystem`, its sign flips with its own state, not with the weather or the
time of day.

</div>

In [ ]:
# OpenDSSDirect.py is a singleton engine (Chapter 1's own note on
# `base_peak_loss_kw` says the same thing): building a second Circuit below
# Clear()s and recompiles the *same* global state, so the first Circuit
# object would silently start reporting the second network's results if
# queried afterward. Reading each result immediately after its own solve,
# into a plain variable, avoids that rather than holding both handles and
# comparing them later.
discharging = build_pv_stress_network(pv_kva=0)
discharging.command(
    "New Storage.Batt1 phases=1 bus1=C.1 kv=0.23 kwrated=3 kwhrated=10 kwhstored=8 %reserve=20 state=DISCHARGING pf=1"
)
discharging.solve()
discharging_state = next(discharging.storage_units).state
discharging_p_kw = discharging.element_powers("storages")["p_kw"].iloc[0]

charging = build_pv_stress_network(pv_kva=0)
charging.command(
    "New Storage.Batt1 phases=1 bus1=C.1 kv=0.23 kwrated=3 kwhrated=10 kwhstored=2 %reserve=20 state=CHARGING pf=1"
)
charging.solve()
charging_state = next(charging.storage_units).state
charging_p_kw = charging.element_powers("storages")["p_kw"].iloc[0]

print(f"Discharging: state={discharging_state}, P={discharging_p_kw:+.2f} kW (negative = supplying the network)")
print(f"Charging:    state={charging_state}, P={charging_p_kw:+.2f} kW (positive = drawing from the network)")

Discharging: state=Discharging, P=-3.00 kW (negative = supplying the network)
Charging:    state=Charging, P=+3.00 kW (positive = drawing from the network)


<div style="text-align: justify;">

## 6. A PV hosting-capacity study

Every piece above comes together here: real load data becomes LoadShapes on
the real 31-customer AusNet feeder, a share of customers get real PV
profiles, and the network solves across a full day at increasing PV
penetration until voltage actually breaches its statutory limit, the same
methodology Team-Nando's own hosting-capacity tutorials use.

</div>

In [ ]:
def build_ausnet_network() -> Circuit:
    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit


circuit = build_ausnet_network()
print(circuit.summary())

{'converged': True, 'n_buses': 61, 'n_lines': 59, 'n_loads': 31, 'n_transformers': 1, 'total_p_loss_kw': 0.5000000340400017, 'total_q_loss_kvar': 0.0003361050336031948}


<div style="text-align: justify;">

`run_penetration` allocates one of the 342 real customer profiles to each
of the network's 31 loads (there are more real customers than houses on
this particular feeder, so each run samples a fresh 31), gives a chosen
percentage of them a real PV system sized at 5kVA, and solves the whole day
half-hour by half-hour, recording the worst (max) bus voltage anywhere on
the feeder at each step.

</div>

In [ ]:
def run_penetration(circuit_builder, penetration: int, pv_kva: float = 5.0, day_idx: int = 363, seed: int = 42):
    rng = np.random.default_rng(seed)
    circuit = circuit_builder()
    loads = list(circuit.loads)

    customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
    n_with_pv = round(len(loads) * penetration / 100)
    pv_customers = set(rng.choice([load.name for load in loads], size=n_with_pv, replace=False))

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")

        if load.name in pv_customers:
            circuit.command(
                f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
                "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
            )
            pv_profile = pv_data[day_idx, :]
            circuit.command(
                f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
            )
            circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

    voltage_rows = []
    for step in circuit.solve_daily(steps=48):
        voltages = circuit.bus_voltages()
        voltage_rows.append(
            {
                "step": step,
                "penetration": penetration,
                "vmax_pu": voltages["vmag_pu"].max(),
                "vmin_pu": voltages["vmag_pu"].min(),
            }
        )
    return pd.DataFrame(voltage_rows)

<div style="text-align: justify;">

Day 363 is the sunniest day in the dataset (the highest normalised PV
profile of the year), the worst case for a hosting-capacity study: if a
penetration level is going to cause a violation, a low-demand, high-sun day
is where it shows up first.

</div>

In [ ]:
penetration_levels = [0, 20, 40, 60, 80, 100]
hosting_capacity_df = pd.concat(
    [run_penetration(build_ausnet_network, pen) for pen in penetration_levels], ignore_index=True
)

summary = hosting_capacity_df.groupby("penetration")["vmax_pu"].max()
print(summary.round(4))

penetration
0      1.0823
20     1.0876
40     1.0956
60     1.0973
80     1.1047
100    1.1046
Name: vmax_pu, dtype: float64


In [ ]:
from lets_plot import geom_hline, geom_line, geom_point, geom_rect

from ark.plot.tokens import DANGER, SUCCESS

LOW, HIGH = 0.94, 1.10


def plot_hosting_capacity(df: pd.DataFrame) -> None:
    envelope = df.groupby("penetration", as_index=False).agg(vmax_pu=("vmax_pu", "max"), vmin_pu=("vmin_pu", "min"))
    compliant_band = pd.DataFrame({"xmin": [envelope["penetration"].min()], "xmax": [envelope["penetration"].max()]})

    plot = (
        ggplot()
        + geom_rect(aes(xmin="xmin", xmax="xmax"), data=compliant_band, ymin=LOW, ymax=HIGH, fill=SUCCESS, alpha=0.08)
        + geom_hline(yintercept=HIGH, linetype="dashed", color=DANGER, size=0.8)
        + geom_hline(yintercept=LOW, linetype="dashed", color=DANGER, size=0.8)
        + geom_line(aes(x="penetration", y="vmax_pu"), data=envelope, color=DANGER, size=1)
        + geom_point(aes(x="penetration", y="vmax_pu"), data=envelope, color=DANGER, size=3)
        + geom_line(aes(x="penetration", y="vmin_pu"), data=envelope, color=PRIMARY, size=1)
        + geom_point(aes(x="penetration", y="vmin_pu"), data=envelope, color=PRIMARY, size=3)
        + labs(
            x="PV penetration (% of customers)",
            y="Voltage (pu)",
            title="PV hosting capacity: worst-case feeder voltage vs. penetration (sunniest day)",
        )
        + modern_theme()
        + ggsize(650, 350)
    )
    return plot


plot_hosting_capacity(hosting_capacity_df)

<div style="text-align: justify;">

The feeder-wide maximum stays inside the compliant band through 60%
penetration, then crosses the 1.10 pu statutory limit at 80%. This is a real
result on real data, not a theoretical ceiling: a specific, reproducible
answer to "how much rooftop solar can this feeder actually take," on the
sunniest day it will ever see, with realistically-sized 5kVA systems. A
utility engineer asking the same question on a different feeder would run
exactly this notebook, swap the network files, and get their own answer.

</div>

<div style="text-align: justify;">

## Where this leaves Part 4

Chapter 1 built the vocabulary: a network, a solve, a per-unit voltage
reading. This chapter turned that into something closer to how a real
utility studies a feeder, real smart-meter data driving a real time series,
DER models with actual physics behind them, and a hosting-capacity result
with a real number attached. The four threads Part 4 opens up next, phase
detection, customer segmentation, voltage-anomaly detection, and DER-forecast
grid health, all build directly on the same `ark.dss.Circuit` this notebook
just extended.

</div>

<div style="text-align: justify;">

## References

</div>